# 2. Inputs and runs

Pipeline inputs answer a harder question than a single procedure call: *which records, shared artifacts, labels, and resolver result formed this run?* The library freezes that answer before planning tasks.


## Records, shared inputs, input sets, and resolvers

An `InputRecord` is one unit of repeated work: a stable key, named artifact identities, and JSON labels. Shared inputs apply to every record. An immutable `InputSet` is a reusable stored collection. An `InputRecordResolver` produces records from external configuration. In every case the result becomes a `RunInputSnapshot`; execution never consults a live resolver again.


In [ ]:
from provium_pipeline import InputRecord, InputRecordKey

records = (
    InputRecord(
        key=InputRecordKey('chapter-1'),
        inputs={'document': ('sha256:document-one',)},
        labels={'language': 'en', 'priority': 1},
    ),
    InputRecord(
        key=InputRecordKey('chapter-2'),
        inputs={'document': ('sha256:document-two',)},
        labels={'language': 'en', 'priority': 2},
    ),
)
assert [str(record.key) for record in records] == ['chapter-1', 'chapter-2']
records


## Snapshot validation and identity

Validation checks required input names, per-record versus shared scope, cardinality, duplicate record keys, and artifact compatibility. Canonical serialization and digests make ordering and equivalent documents deterministic. Resolver configuration and result digests are retained so an external query can be audited later.

A run fingerprint combines the compiled pipeline digest, resolved configuration digest, input snapshot digest, and relevant metadata/idempotency fields. Repeating the same idempotency key with the same request returns the original run; reusing it for different semantics is a conflict.


In [ ]:
from provium_pipeline import (
    CreateRunRequest,
    RunInputSnapshot,
    RunPlan,
    RunState,
    plan_run,
    resolve_input_snapshot,
    run_fingerprint,
)

run_contract = {
    'snapshot': RunInputSnapshot, 'request': CreateRunRequest,
    'planner': plan_run, 'plan': RunPlan, 'fingerprint': run_fingerprint,
    'resolver': resolve_input_snapshot,
}
assert RunState.PLANNED.value == 'planned'
run_contract


## Task planning

Planning expands the compiled graph across records. Record-scoped nodes normally create one task per record; shared work can create shared tasks. Each task records dependency task IDs and expected outputs. The plan is deterministic, so storage can enforce idempotency and workers never need to rediscover topology.

CLI path: create an input set with `provium input-set create records.ndjson --identifier tutorial.batch`, then create a durable run with `provium run create PIPELINE --input-set INPUT_SET_ID`. Resolver-backed workflows use `provium pipeline enqueue PIPELINE --records-from RESOLVER_SPEC`.

**What to notice:** resolvers discover candidates, snapshots preserve evidence, runs freeze intent, and task plans turn graph semantics into schedulable units. Next: [dispatch and execution](03-dispatch-and-execution.ipynb). Reference: [inputs and runs](../docs/inputs-and-runs.md).
